# YOLOv8s Drone Detection Training

**Dataset**: 1,359 images, 1 class (drone)
**Training Time**: ~6-10 hours on Mac M4

## 0. Setup Display

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✓ Display setup complete')

✓ Display setup complete


## 1. Install Ultralytics

In [2]:
!pip install ultralytics


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Load YOLOv8s Model

In [3]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO('yolov8s.pt')

print('✓ YOLOv8s model loaded')

✓ YOLOv8s model loaded


## 3. Train Model

Optimized for Mac M4: batch=16, MPS GPU, RAM caching

In [4]:
# Training configuration
results = model.train(
    data='/Users/hilkin/Development/Drone-Detection/drone_dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,         # Optimized for 25GB RAM
    device='mps',     # Mac M4 GPU
    workers=8,        # Parallel data loading
    cache='ram',      # Cache dataset in RAM
    project='yolov8s_drone_results',
    name='training',
    patience=20,      # Early stopping
    save_period=10,   # Save checkpoint every 10 epochs
    amp=True,         # Mixed precision
    plots=True
)

print('\n✓ Training complete!')

New https://pypi.org/project/ultralytics/8.4.14 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.13 🚀 Python-3.14.2 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/hilkin/Development/Drone-Detection/drone_dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=trai

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


train: Caching images (0.7GB RAM): 85% ━━━━━━━━━━── 861/1012 1.3Kit/s 0.5s<0.1s

libpng warning: iCCP: known incorrect sRGB profile


train: Caching images (0.8GB RAM): 100% ━━━━━━━━━━━━ 1012/1012 1.1Kit/s 0.9s0.1s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 168.3±124.8 MB/s, size: 41.1 KB)
val: Scanning /Users/hilkin/Development/Drone-Detection/drone-dataset-(uav)-DatasetNinja/valid/labels.cache... 347 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 347/347 181.9Mit/s 0.0s
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.3GB RAM): 100% ━━━━━━━━━━━━ 347/347 1.1Kit/s 0.3s<0.5s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with paramet

KeyboardInterrupt: 

## 4. Evaluate Model

In [ ]:
# Evaluate on validation set
metrics = model.val()

print(f'\nmAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall: {metrics.box.mr:.4f}')

## 5. Test Predictions (with Image Display)

In [ ]:
import glob
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Get validation images
val_images = glob.glob('/Users/hilkin/Development/Drone-Detection/drone-dataset-(uav)-DatasetNinja/valid/images/*')[:6]

# Run predictions
results = model.predict(val_images, conf=0.25, save=False)

# Display with larger figure
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, result in enumerate(results):
    # Get plotted image (BGR format from ultralytics)
    img_bgr = result.plot()
    
    # Convert BGR to RGB for matplotlib
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Display
    axes[idx].imshow(img_rgb)
    axes[idx].axis('off')
    axes[idx].set_title(f'Image {idx+1} - {len(result.boxes)} drone(s)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('✓ Predictions displayed above')

## 6. Show Training Results

In [ ]:
import os
from IPython.display import Image as IPImage, display

# Display training results plot
results_img = 'yolov8s_drone_results/training/results.png'

if os.path.exists(results_img):
    print("Training Results:")
    display(IPImage(filename=results_img, width=1000))
else:
    print("Results image not found yet. Continue training!")

## 7. Show Confusion Matrix

In [ ]:
# Display confusion matrix
cm_img = 'yolov8s_drone_results/training/confusion_matrix.png'

if os.path.exists(cm_img):
    print("Confusion Matrix:")
    display(IPImage(filename=cm_img, width=600))
else:
    print("Confusion matrix not found yet.")

## 8. Export Model

In [ ]:
# Export to different formats
model.export(format='onnx')    # Universal
model.export(format='coreml')  # Apple devices

print('✓ Model exported!')

## 9. Test on Your Own Image (Optional)

In [ ]:
# Upload or specify path to your test image
test_image_path = '/Users/hilkin/Development/Drone-Detection/pexels.jpeg'  # Change this path

if os.path.exists(test_image_path):
    # Run prediction
    result = model.predict(test_image_path, conf=0.25)[0]
    
    # Display
    plt.figure(figsize=(12, 8))
    img_bgr = result.plot()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'Detected {len(result.boxes)} drone(s)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f'✓ Found {len(result.boxes)} drone(s) in the image')
else:
    print(f'Image not found: {test_image_path}')
    print('Update test_image_path variable to point to your image')

## Summary

**Model Location**: `yolov8s_drone_results/training/weights/best.pt`

**Performance Metrics**: Check output above

**Next Steps**:
1. Use `best.pt` for inference on new images
2. Deploy to your application
3. Try RT-DETR notebook for even higher accuracy

**To use your model later**:
```python
from ultralytics import YOLO
model = YOLO('yolov8s_drone_results/training/weights/best.pt')
results = model.predict('your_image.jpg')
```